# RAG Assignment: Comparing Retrieval and Re-Ranking Methods

**Document:** the EU General Data Protection Regulation (GDPR, Regulation (EU) 2016/679), the official 88-page PDF.
All pipelines search the **same chunked corpus** and use the **same LLM prompt**.

| | Pipeline 1 - Dense baseline | Pipeline 2 - Dense + Re-Ranking | Pipeline 3 - BM25 (keyword) |
|---|---|---|---|
| Retrieve | Bi-Encoder `all-MiniLM-L6-v2` + FAISS, Top-10 by cosine similarity | the same Top-10 as Pipeline 1 | BM25 keyword search, Top-10 by BM25 score |
| Re-rank | - | Cross-Encoder `ms-marco-MiniLM-L-6-v2` scores all 10 (query, chunk) pairs, sort descending | - |
| To the LLM | Top-3 by cosine -> **Answer 1** | Top-3 by Cross-Encoder score -> **Answer 2** | Top-3 by BM25 -> **Answer 3** |

LLM: `Qwen2.5-3B-Instruct`, a local open-weights model (Hugging Face), run on CPU with greedy decoding.

**Outline:** 1. Setup - 2. Load the PDF - 3. Chunking - 4. Pipeline 1 - 5. Pipeline 2 - 6. Pipeline 3 (BM25) - 7. LLM - 8. Failure-mode analysis - 9. Rank-shift summary - 10. Latency profiling - 11. Try more questions

> The notebook loads `gdpr.pdf` from the project folder.

## 1. Setup

In [1]:
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import pymupdf
import torch
from IPython.display import Markdown, display
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

PDF_PATH = Path("gdpr.pdf")
RESULTS_DIR = Path("untracked/results")   # saved results (git-ignored)
CACHE_DIR = Path("untracked/cache")   # chunks + FAISS index are saved here after the first run (git-ignored)
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # dense Bi-Encoder
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # re-ranker named in the assignment
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"                        # local open-weights LLM
TOP_K = 10             # candidates retrieved by the Bi-Encoder
TOP_N = 3              # chunks given to the LLM
MAX_CHUNK_CHARS = 900  # chunk body size limit
MAX_NEW_TOKENS = 200

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(0)
pd.set_option("display.max_colwidth", 80)
assert PDF_PATH.exists(), "gdpr.pdf not found in the project folder."

## 2. Load the PDF

We read the PDF with PyMuPDF and keep only the regulation text:

* the page header on every page (`4.5.2016`, `EN`, `Official Journal of the European Union`, `L 119/33`) is removed;
* footnotes are removed using their smaller font size (8.5 pt, while the body text is 9.6 pt);
* words split across lines with a soft hyphen are joined again.

In [2]:
RUNNING_HEADER = re.compile(r"^(\d{1,2}\.\d{1,2}\.\d{4}|EN|Official Journal of the European Union|L \d+/\d+)$")
BODY_FONT_MIN = 9.0    # body text is 9.6 pt; footnotes are 8.5 pt
SUPERSCRIPT_MAX = 7.0  # footnote reference markers are ~5 pt
STRUCTURAL = re.compile(r"^(HAVE ADOPTED THIS REGULATION:|CHAPTER [IVX]+)$")


def load_pdf_lines(path):
    # Return the clean text lines of the PDF, in reading order.
    lines = []
    with pymupdf.open(path) as doc:
        n_pages = len(doc)
        for page in doc:
            for block in page.get_text("dict")["blocks"]:
                for line in block.get("lines", []):
                    spans = [sp for sp in line["spans"] if sp["size"] >= SUPERSCRIPT_MAX]
                    text = "".join(sp["text"] for sp in spans).strip()
                    if not text or RUNNING_HEADER.match(text):
                        continue
                    # a line set entirely in the small font is a footnote (unless it is a structural heading)
                    if not any(sp["size"] >= BODY_FONT_MIN for sp in spans) and not STRUCTURAL.match(text):
                        continue
                    lines.append(text)
    joined = "\n".join(lines).replace("­\n", "").replace("­", "")  # re-join soft-hyphenated words
    return joined.split("\n"), n_pages

## 3. Chunking

The GDPR has **173 recitals** (explanations) and **99 articles** (the binding rules). We first split the text into these sections, so a chunk never mixes two articles.
Then each section is cut into chunks:

* an article is split at its numbered paragraphs (`1.`, `2.`, ...), and short paragraphs are packed together up to ~900 characters;
* every chunk starts with a header such as `GDPR Article 33 - Notification of a personal data breach ...`, so it keeps its topic;
* every chunk gets a short reference like `Art. 33(1-2)` or `Recital 85`, used in the tables below.

In [3]:
@dataclass
class Section:
    kind: str    # "recital" or "article"
    number: int
    title: str
    body: str


def split_sections(lines):
    idx = lines.index("HAVE ADOPTED THIS REGULATION:")  # recitals end here, articles start after it
    pre, post = lines[:idx], lines[idx + 1:]
    sections = []

    # ---- recitals: "(n) text..." with strictly increasing n
    cur, buf, expected = None, [], 1
    for ln in pre:
        m = re.match(r"^\((\d{1,3})\)\s*(.*)$", ln)
        if m and int(m.group(1)) == expected:
            if cur is not None:
                sections.append(Section("recital", cur, f"Recital {cur}", " ".join(buf)))
            cur, buf, expected = expected, ([m.group(2)] if m.group(2) else []), expected + 1
        elif cur is not None:
            buf.append(ln)
    sections.append(Section("recital", cur, f"Recital {cur}", " ".join(buf)))

    # ---- articles: "Article N" line, then a title line, then the body
    i, cur, title, buf = 0, None, None, []
    while i < len(post):
        ln = post[i]
        if ln.startswith("This Regulation shall be binding in its entirety"):  # closing formula
            break
        if re.match(r"^(CHAPTER [IVX]+|Section \d+)$", ln):  # chapter/section heading + its title line
            i += 2
            continue
        m = re.match(r"^Article (\d{1,3})$", ln)
        if m:
            if cur is not None:
                sections.append(Section("article", cur, title, "\n".join(buf)))
            cur, title, buf = int(m.group(1)), post[i + 1], []
            t = re.match(r"^(.*?)\s+(\d+\.)$", title)  # rare case: "Title 1." merged onto one line
            if t:
                title, buf = t.group(1), [t.group(2)]
            i += 2
            continue
        if cur is not None:
            buf.append(ln)
        i += 1
    sections.append(Section("article", cur, title, "\n".join(buf)))
    return sections

In [4]:
def _flow(lines):
    # Join hard-wrapped PDF lines into one paragraph.
    return re.sub(r"\s+", " ", " ".join(lines)).strip()


def _split_long(text, max_chars):
    # Split an over-long paragraph on sentence / list-point boundaries and pack the pieces greedily.
    pieces = re.split(r"(?<=[.;:])\s+(?=\(?[A-Z(])", text)
    out, cur = [], ""
    for p in pieces:
        if cur and len(cur) + 1 + len(p) > max_chars:
            out.append(cur)
            cur = p
        else:
            cur = f"{cur} {p}".strip()
    if cur:
        out.append(cur)
    return out


def _make_chunk(sec, header, body, paras, chunk_id):
    if sec.kind == "article":
        span = "" if not paras else (f"({paras[0]})" if len(paras) == 1 else f"({paras[0]}-{paras[-1]})")
        ref = f"Art. {sec.number}{span}"
    else:
        ref = f"Recital {sec.number}"
    return {"chunk_id": chunk_id, "ref": ref, "kind": sec.kind, "number": sec.number,
            "title": sec.title, "text": f"{header}\n{body}"}


def chunk_sections(sections, max_chars=MAX_CHUNK_CHARS):
    chunks = []
    for sec in sections:
        if sec.kind == "recital":
            units = [(None, sec.body.strip())]
            header = f"GDPR Recital {sec.number}"
        else:
            # numbered paragraphs begin with a line that is just "3."
            units, cur_no, cur = [], None, []
            for ln in sec.body.split("\n"):
                m = re.match(r"^(\d{1,2})\.$", ln)
                if m:
                    if cur:
                        units.append((cur_no, _flow(cur)))
                    cur_no, cur = int(m.group(1)), []
                else:
                    cur.append(ln)
            if cur:
                units.append((cur_no, _flow(cur)))
            header = f"GDPR Article {sec.number} - {sec.title}"

        # explode long paragraphs, then greedily pack consecutive pieces up to max_chars
        pieces = []
        for no, txt in units:
            label = f"{no}. " if no is not None else ""
            for j, part in enumerate(_split_long(txt, max_chars)):
                pieces.append((no, (label if j == 0 else "") + part))
        buf, paras = "", []
        for no, txt in pieces:
            if buf and len(buf) + 1 + len(txt) > max_chars:
                chunks.append(_make_chunk(sec, header, buf, paras, len(chunks)))
                buf, paras = "", []
            buf = f"{buf}\n{txt}".strip()
            if no is not None and no not in paras:
                paras.append(no)
        if buf:
            chunks.append(_make_chunk(sec, header, buf, paras, len(chunks)))
    return chunks

**Build the chunks once and store them.** The first run reads the PDF, chunks it and saves the chunks to `untracked/cache/chunks.json`.
Later runs just load that file, so the PDF is not processed again.

In [5]:
CHUNKS_FILE = CACHE_DIR / "chunks.json"
if CHUNKS_FILE.exists():
    chunks = json.loads(CHUNKS_FILE.read_text())
    print(f"Loaded {len(chunks)} chunks from {CHUNKS_FILE}")
else:
    lines, n_pages = load_pdf_lines(PDF_PATH)
    sections = split_sections(lines)
    assert [s.number for s in sections if s.kind == "recital"] == list(range(1, 174)), "expected recitals 1..173"
    assert [s.number for s in sections if s.kind == "article"] == list(range(1, 100)), "expected articles 1..99"
    chunks = chunk_sections(sections)
    CHUNKS_FILE.write_text(json.dumps(chunks, ensure_ascii=False))
    print(f"{n_pages} pages -> 173 recitals + 99 articles -> {len(chunks)} chunks, saved to {CHUNKS_FILE}")

chunk_df = pd.DataFrame(chunks)
chunk_texts = chunk_df["text"].tolist()
print(f"median chunk length: {int(chunk_df.text.str.len().median())} characters")
print("example chunk:\n" + chunk_texts[0][:300])

Loaded 559 chunks from untracked/cache/chunks.json
median chunk length: 673 characters
example chunk:
GDPR Recital 1
The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1) of the Charter of Fundamental Rights of the European Union (the ‘Charter’) and Article 16(1) of the Treaty on the Functioning of the European Union (TFEU) provide that


## 4. Pipeline 1 - Baseline retrieval (Bi-Encoder)

a. Every chunk is encoded **once** into a 384-number vector by the Bi-Encoder and stored in a FAISS vector index (saved to `untracked/cache/gdpr.faiss`, so later runs only load it).
Vectors are normalised, so the inner product used by `IndexFlatIP` equals **cosine similarity**.

b. For a query, we encode it the same way and retrieve the **Top 10** chunks.

c. The **Top 3** of those go straight to the LLM to produce **Answer 1**.

In [6]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")   # also needed to encode each query

INDEX_FILE = CACHE_DIR / "gdpr.faiss"
if INDEX_FILE.exists():
    index = faiss.read_index(str(INDEX_FILE))      # load the stored vector index
    print(f"Loaded vector store from {INDEX_FILE}")
else:
    chunk_emb = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    index = faiss.IndexFlatIP(chunk_emb.shape[1])  # exact inner-product (= cosine) search
    index.add(chunk_emb)
    faiss.write_index(index, str(INDEX_FILE))      # store it for the next run
    print(f"Built vector store and saved it to {INDEX_FILE}")
assert index.ntotal == len(chunk_texts), "cache is out of date - delete untracked/cache/ and re-run"
print(f"Vector store: {index.ntotal} chunks x {index.d} dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from untracked/cache/gdpr.faiss
Vector store: 559 chunks x 384 dimensions


In [7]:
def baseline_retrieve(query, k=TOP_K):
    # Stage 1: embed the query, return the top-k chunk ids and their cosine similarities.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True,
                          show_progress_bar=False).astype(np.float32)
    sims, ids = index.search(q, k)
    return ids[0].tolist(), sims[0].tolist()

## 5. Pipeline 2 - Re-ranking (Cross-Encoder)

a. Take the **same Top 10** candidates from the baseline.

b. Pass all 10 `(query, chunk)` pairs through the Cross-Encoder. It reads the query and the chunk **together** and gives one relevance score per pair.

c. Sort the 10 candidates by that score (highest first) and keep the **Top 3**.

d. Give those 3 chunks to the LLM, with the identical prompt, to produce **Answer 2**.

In [8]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")
print(f"Cross-Encoder max_seq_length={cross_encoder.max_seq_length}")


def rerank(query, candidate_ids):
    # Stage 2: score every (query, chunk) pair jointly and sort descending by Cross-Encoder score.
    scores = cross_encoder.predict([(query, chunk_texts[i]) for i in candidate_ids], show_progress_bar=False)
    order = np.argsort(-scores)
    return [candidate_ids[j] for j in order], scores[order].tolist()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder max_seq_length=512


## 6. Pipeline 3 - BM25 keyword retrieval

A second, different **retrieval** method. BM25 is a classic **sparse / keyword** method: it does not use a neural model.
It scores each chunk by how often the question's words appear in it, giving rare words more weight than common ones.

a. Split every chunk into lower-case words and build a BM25 index (it takes well under a second, so it is simply rebuilt from the cached chunks).

b. For a query, score all chunks and take the **Top 10**.

c. The **Top 3** go to the LLM, with the identical prompt, to produce **Answer 3**.

In [9]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())   # simple word split, lower-case


bm25 = BM25Okapi([tokenize(t) for t in chunk_texts])


def bm25_retrieve(query, k=TOP_K):
    # Pipeline 3: Top-k chunks by BM25 keyword score.
    scores = bm25.get_scores(tokenize(query))
    ids = np.argsort(-scores)[:k].tolist()
    return ids, scores[ids].tolist()

## 7. LLM and the shared prompt

All pipelines call the same `generate_answer()` with the same prompt; only the 3 context chunks are different.
Greedy decoding (`do_sample=False`) makes the output deterministic, so any difference between the answers comes from the retrieved chunks.

In [10]:
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float32)
llm.eval()

SYSTEM_PROMPT = (
    "You are a careful legal assistant answering questions about the EU General Data Protection Regulation (GDPR). "
    "Answer ONLY from the numbered context passages. State the exact figures, conditions and exceptions they contain "
    "and cite the passage reference in brackets, e.g. [Art. 33(1)]. If the passages do not contain the answer, reply: "
    "\"The provided context does not answer this question.\" Keep the answer under 120 words."
)


def build_prompt(query, context_ids):
    context = "\n\n".join(f"[{chunk_df.ref[i]}]\n{chunk_texts[i]}" for i in context_ids)
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context passages:\n\n{context}\n\nQuestion: {query}"}]


@torch.inference_mode()
def generate_answer(query, context_ids):
    messages = build_prompt(query, context_ids)
    inputs = llm_tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True)
    t0 = time.perf_counter()
    out = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                       pad_token_id=llm_tokenizer.eos_token_id)
    elapsed = time.perf_counter() - t0
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return {"answer": llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip(),
            "gen_s": elapsed, "prompt_tokens": int(inputs["input_ids"].shape[1]),
            "new_tokens": int(len(new_tokens))}

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [11]:
def run_pipelines(query):
    # Pipelines 1, 2 and 3 for one query.
    base_ids, base_sims = baseline_retrieve(query)   # Top-10 from the Bi-Encoder (Pipelines 1 and 2)
    rr_ids, rr_scores = rerank(query, base_ids)      # the same 10, re-sorted by the Cross-Encoder (Pipeline 2)
    bm_ids, bm_scores = bm25_retrieve(query)         # Top-10 from BM25 (Pipeline 3)
    return {"query": query, "base_ids": base_ids, "base_sims": base_sims, "rr_ids": rr_ids, "rr_scores": rr_scores,
            "bm_ids": bm_ids, "bm_scores": bm_scores,
            "p1": generate_answer(query, base_ids[:TOP_N]),   # Answer 1
            "p2": generate_answer(query, rr_ids[:TOP_N]),     # Answer 2
            "p3": generate_answer(query, bm_ids[:TOP_N])}     # Answer 3

## 8. Failure-mode analysis

Five queries covering the failure types in the assignment. Each has a **gold chunk**: the chunk that contains the exact sentence of the regulation that answers it (found by a verbatim phrase from the law).

| ID | Failure type | Query |
|---|---|---|
| Q1 | Keyword trap / distractor | What is the highest penalty for a company that violates people's rights under GDPR? |
| Q2 | Exception / corner case | Do I have to notify affected users of a breach if the stolen data was encrypted? |
| Q3 | Negation / role constraint | Can the data protection officer be fired for doing their job? |
| Q4 | Other - vital chunk buried at rank 4-10 | Does a company have to stop processing if a person objects? |
| Q5 | Keyword trap / synonym | How quickly must a company fix wrong information about me? |

For each query the table shows the **Bi-Encoder, Cross-Encoder and BM25 rankings side by side**; the gold chunk is marked with `<< GOLD`, and the line marks the Top-3 that the LLM sees.

In [12]:
FAILURE_QUERIES = [
    {"id": "Q1", "type": "Keyword trap / distractor",
     "query": "What is the highest penalty for a company that violates people's rights under GDPR?",
     "gold": "the data subjects' rights pursuant to Articles 12 to 22",
     "correct": "Art. 83(5)(b): fines up to EUR 20 million or 4% of worldwide annual turnover, whichever is higher. "
                "Distractors: Art. 84 is titled 'Penalties' but only lets Member States set other penalties; "
                "Art. 83(4) is the lower EUR 10 million / 2% tier."},
    {"id": "Q2", "type": "Exception / corner case",
     "query": "Do I have to notify affected users of a breach if the stolen data was encrypted?",
     "gold": "unintelligible to any person who is not authorised",
     "correct": "Art. 34(3)(a): no - telling the users is not required if the data were made unintelligible "
                "to unauthorised persons, such as by encryption."},
    {"id": "Q3", "type": "Negation / role constraint",
     "query": "Can the data protection officer be fired for doing their job?",
     "gold": "shall not be dismissed or penalised",
     "correct": "Art. 38(3): no - the DPO shall not be dismissed or penalised for performing their tasks."},
    {"id": "Q4", "type": "Other - buried at rank 4-10",
     "query": "Does a company have to stop processing if a person objects?",
     "gold": "compelling legitimate grounds",
     "correct": "Art. 21(1): yes, it must stop, unless it shows compelling legitimate grounds that override the "
                "person's interests, or the processing is needed for legal claims."},
    {"id": "Q5", "type": "Keyword trap / synonym",
     "query": "How quickly must a company fix wrong information about me?",
     "gold": "rectification of inaccurate personal data concerning him or her",
     "correct": "Art. 16: the person has the right to have inaccurate data rectified (corrected) 'without undue delay'. "
                "Trap: 'how quickly' matches the 72-hour data-breach rules, which are about a different duty."},
]


def gold_ids(phrase):
    # Chunks containing the verbatim gold phrase (case/whitespace-insensitive).
    norm = lambda s: re.sub(r"\s+", " ", s).lower()
    return {i for i, t in enumerate(chunk_texts) if norm(phrase) in norm(t)}


def rank_of(ids, gold):
    return next((r + 1 for r, i in enumerate(ids) if i in gold), None)


def side_by_side(res, gold):
    # Bi-Encoder, Cross-Encoder and BM25 rankings next to each other.
    mark = lambda i: f"{chunk_df.ref[i]}  << GOLD" if i in gold else chunk_df.ref[i]
    df = pd.DataFrame({
        "P1 Bi-Encoder": [mark(i) for i in res["base_ids"]],
        "cosine": np.round(res["base_sims"], 3),
        "P2 Cross-Encoder": [mark(i) for i in res["rr_ids"]],
        "CE score": np.round(res["rr_scores"], 2),
        "P3 BM25": [mark(i) for i in res["bm_ids"]],
        "BM25 score": np.round(res["bm_scores"], 1),
    }, index=pd.Index(range(1, TOP_K + 1), name="rank"))
    return df.style.format({"cosine": "{:.3f}", "CE score": "{:.2f}", "BM25 score": "{:.1f}"}).set_table_styles([{"selector": "tbody tr:nth-child(3) td, tbody tr:nth-child(3) th",
                                       "props": "border-bottom: 2px solid #888;"}])


results = []
for fq in FAILURE_QUERIES:
    gold = gold_ids(fq["gold"])
    assert gold, f"gold phrase not found: {fq['gold']}"
    res = run_pipelines(fq["query"]) | fq
    res["gold_ids"] = gold
    res["gold_ref"] = ", ".join(sorted({chunk_df.ref[i] for i in gold}))
    res["base_rank"], res["rr_rank"] = rank_of(res["base_ids"], gold), rank_of(res["rr_ids"], gold)
    res["bm_rank"] = rank_of(res["bm_ids"], gold)   # None = not in BM25's Top-10
    results.append(res)
print("done")

done


In [13]:
def show(res):
    display(Markdown(
        f"### {res['id']} - {res['type']}\n"
        f"**Query:** {res['query']}  \n"
        f"**Gold chunk:** {res['gold_ref']} - rank: **Bi-Encoder #{res['base_rank']}, Cross-Encoder #{res['rr_rank']}, "
        f"BM25 {'#' + str(res['bm_rank']) if res['bm_rank'] else 'not in Top-10'}**  \n"
        f"**Correct answer:** {res['correct']}"))
    display(side_by_side(res, res["gold_ids"]))
    display(Markdown(
        f"**Answer 1** (Bi-Encoder Top-3: {', '.join(chunk_df.ref[i] for i in res['base_ids'][:TOP_N])})\n\n"
        f"> {res['p1']['answer']}\n\n"
        f"**Answer 2** (Cross-Encoder Top-3: {', '.join(chunk_df.ref[i] for i in res['rr_ids'][:TOP_N])})\n\n"
        f"> {res['p2']['answer']}\n\n"
        f"**Answer 3** (BM25 Top-3: {', '.join(chunk_df.ref[i] for i in res['bm_ids'][:TOP_N])})\n\n"
        f"> {res['p3']['answer']}"))


show(results[0])

### Q1 - Keyword trap / distractor
**Query:** What is the highest penalty for a company that violates people's rights under GDPR?  
**Gold chunk:** Art. 83(5) - rank: **Bi-Encoder #4, Cross-Encoder #1, BM25 not in Top-10**  
**Correct answer:** Art. 83(5)(b): fines up to EUR 20 million or 4% of worldwide annual turnover, whichever is higher. Distractors: Art. 84 is titled 'Penalties' but only lets Member States set other penalties; Art. 83(4) is the lower EUR 10 million / 2% tier.

,P1 Bi-Encoder,cosine,P2 Cross-Encoder,CE score,P3 BM25,BM25 score
rank,,,,,,
1,Art. 84(1-2),0.704,Art. 83(5) << GOLD,2.53,Recital 56,17.3
2,Art. 83(4),0.661,Art. 83(4),1.57,Art. 18(2-3),12.9
3,Recital 150,0.640,Recital 150,0.97,Recital 151,12.9
4,Art. 83(5) << GOLD,0.639,Art. 84(1-2),0.83,Recital 142,12.6
5,Art. 83(2-3),0.633,Recital 148,0.71,Recital 39,12.4
6,Recital 148,0.627,Art. 83(6-8),0.35,Art. 80(2),12.3
7,Art. 83(6-8),0.626,Art. 83(2-3),-0.52,Art. 12(2),12.0
8,Art. 83(2),0.615,Recital 148,-0.61,Recital 157,12.0
9,Art. 47(2),0.609,Art. 47(2),-0.91,Art. 5(1),11.6


**Answer 1** (Bi-Encoder Top-3: Art. 84(1-2), Art. 83(4), Recital 150)

> The highest penalty for a company that violates people's rights under GDPR is 2% of the company's global annual turnover of the previous financial year, whichever is higher. [Art. 83(4)]

**Answer 2** (Cross-Encoder Top-3: Art. 83(5), Art. 83(4), Recital 150)

> The highest penalty for a company violating people's rights under GDPR is 4% of the company's global annual turnover, or up to 20,000,000 EUR, whichever is greater. [Art. 83(5)]

**Answer 3** (BM25 Top-3: Recital 56, Art. 18(2-3), Recital 151)

> The provided context does not answer this question.

In [14]:
show(results[1])

### Q2 - Exception / corner case
**Query:** Do I have to notify affected users of a breach if the stolen data was encrypted?  
**Gold chunk:** Art. 34(3) - rank: **Bi-Encoder #4, Cross-Encoder #1, BM25 #3**  
**Correct answer:** Art. 34(3)(a): no - telling the users is not required if the data were made unintelligible to unauthorised persons, such as by encryption.

,P1 Bi-Encoder,cosine,P2 Cross-Encoder,CE score,P3 BM25,BM25 score
rank,,,,,,
1,Art. 33(1-2),0.482,Art. 34(3) << GOLD,-0.02,Art. 34(4),23.8
2,Recital 87,0.478,Recital 85,-0.22,Art. 33(1-2),22.3
3,Recital 85,0.472,Art. 33(1-2),-0.69,Art. 34(3) << GOLD,20.4
4,Art. 34(3) << GOLD,0.461,Recital 88,-2.45,Recital 87,20.1
5,Art. 34(1-2),0.446,Recital 85,-2.51,Art. 70(1),19.0
6,Art. 33(3-4),0.441,Recital 87,-3.07,Recital 85,18.8
7,Recital 85,0.435,Art. 33(3-4),-3.22,Art. 83(2),18.2
8,Recital 88,0.430,Art. 70(1),-3.44,Art. 34(1-2),17.8
9,Art. 33(5),0.430,Art. 33(5),-3.70,Art. 11(1-2),17.5


**Answer 1** (Bi-Encoder Top-3: Art. 33(1-2), Recital 87, Recital 85)

> The provided context does not directly state whether encryption of data affects the notification obligation. However, it mentions that if the controller can demonstrate that the personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons, they are not required to notify the supervisory authority. [Art. 33(2)]

**Answer 2** (Cross-Encoder Top-3: Art. 34(3), Recital 85, Art. 33(1-2))

> Yes, according to Art. 34(3)(a), if the controller has implemented appropriate technical and organizational measures, such as encryption, and those measures were applied to the personal data affected by the breach, then notification to the data subjects is not required.

**Answer 3** (BM25 Top-3: Art. 34(4), Art. 33(1-2), Art. 34(3))

> Yes, you have to notify affected users if the stolen data was encrypted [Art. 34(3)(a)]. This is because encryption is mentioned as a measure that renders personal data unintelligible to unauthorized individuals, thus meeting the condition for not requiring notification to users.

In [15]:
show(results[2])

### Q3 - Negation / role constraint
**Query:** Can the data protection officer be fired for doing their job?  
**Gold chunk:** Art. 38(1-3) - rank: **Bi-Encoder #5, Cross-Encoder #1, BM25 #5**  
**Correct answer:** Art. 38(3): no - the DPO shall not be dismissed or penalised for performing their tasks.

,P1 Bi-Encoder,cosine,P2 Cross-Encoder,CE score,P3 BM25,BM25 score
rank,,,,,,
1,Art. 38(4-6),0.565,Art. 38(1-3) << GOLD,2.48,Art. 37(3-5),16.8
2,Art. 37(6-7),0.510,Art. 38(4-6),2.11,Art. 38(4-6),16.1
3,Art. 37(1-2),0.497,Art. 37(6-7),1.77,Art. 57(1-3),15.8
4,Art. 39(1),0.490,Art. 39(1),0.86,Art. 37(1-2),15.5
5,Art. 38(1-3) << GOLD,0.469,Art. 39(1-2),0.69,Art. 38(1-3) << GOLD,14.6
6,Art. 37(3-5),0.446,Art. 37(3-5),-0.27,Art. 37(6-7),14.3
7,Recital 87,0.442,Art. 37(1-2),-0.45,Art. 33(3-4),13.1
8,Art. 39(1-2),0.427,Recital 87,-6.66,Recital 66,12.8
9,Art. 34(4),0.421,Art. 34(4),-7.52,Recital 28,12.8


**Answer 1** (Bi-Encoder Top-3: Art. 38(4-6), Art. 37(6-7), Art. 37(1-2))

> According to Art. 38(5), the data protection officer is bound by secrecy concerning the performance of their tasks, in accordance with Union or Member State law. This implies that the officer cannot be fired for doing their job as it would involve breaching confidentiality. [Art. 38(5)]

**Answer 2** (Cross-Encoder Top-3: Art. 38(1-3), Art. 38(4-6), Art. 37(6-7))

> Yes, the data protection officer cannot be dismissed or penalized by the controller or processor for performing their tasks [Art. 38(3)].

**Answer 3** (BM25 Top-3: Art. 37(3-5), Art. 38(4-6), Art. 57(1-3))

> No, the data protection officer cannot be fired for doing their job according to [Art. 38(6)]. This passage states that the controller or processor shall ensure that any tasks and duties of the data protection officer do not result in a conflict of interests. Therefore, the officer's role is protected from being terminated due to the performance of their duties.

In [16]:
show(results[3])

### Q4 - Other - buried at rank 4-10
**Query:** Does a company have to stop processing if a person objects?  
**Gold chunk:** Art. 21(1-2) - rank: **Bi-Encoder #9, Cross-Encoder #1, BM25 not in Top-10**  
**Correct answer:** Art. 21(1): yes, it must stop, unless it shows compelling legitimate grounds that override the person's interests, or the processing is needed for legal claims.

,P1 Bi-Encoder,cosine,P2 Cross-Encoder,CE score,P3 BM25,BM25 score
rank,,,,,,
1,Recital 51,0.486,Art. 21(1-2) << GOLD,-0.47,Recital 65,17.7
2,Recital 81,0.482,Art. 17(1),-4.02,Art. 17(1),16.7
3,Art. 6(1),0.477,Art. 29,-4.25,Recital 127,14.8
4,Art. 29,0.476,Recital 128,-4.31,Art. 78(1-3),14.2
5,Recital 50,0.473,Art. 6(1),-4.99,Recital 143,13.3
6,Recital 128,0.467,Recital 51,-6.85,Recital 18,13.1
7,Art. 88(2-3),0.467,Recital 50,-7.19,Recital 43,12.8
8,Art. 17(1),0.464,Art. 4,-7.43,Recital 45,12.8
9,Art. 21(1-2) << GOLD,0.461,Art. 88(2-3),-8.61,Art. 82(1-3),12.7


**Answer 1** (Bi-Encoder Top-3: Recital 51, Recital 81, Art. 6(1))

> The provided context does not directly state whether a company has to stop processing if a person objects. However, GDPR Article 6(1)(e) states that processing is necessary for the performance of a task carried out in the public interest or in the exercise of official authority vested in the controller. This implies that if an individual objects, the company might need to reconsider the necessity of processing their data for tasks related to public interest or official authority, but it does not explicitly mandate stopping processing. [Art. 6(1)(e)]

**Answer 2** (Cross-Encoder Top-3: Art. 21(1-2), Art. 17(1), Art. 29)

> Yes, if a person objects, the company has to stop processing the personal data unless there are compelling legitimate grounds for the processing that override the interests, rights and freedoms of the data subject [Art. 21(1-2)].

**Answer 3** (BM25 Top-3: Recital 65, Art. 17(1), Recital 127)

> Yes, a company must stop processing if a person objects, as stated in Art. 17(1)(c) [Art. 17(1)] where it says "the data subject objects to the processing pursuant to Article 21(1) and there are no overriding legitimate grounds for the processing".

In [17]:
show(results[4])

### Q5 - Keyword trap / synonym
**Query:** How quickly must a company fix wrong information about me?  
**Gold chunk:** Art. 16 - rank: **Bi-Encoder #9, Cross-Encoder #3, BM25 not in Top-10**  
**Correct answer:** Art. 16: the person has the right to have inaccurate data rectified (corrected) 'without undue delay'. Trap: 'how quickly' matches the 72-hour data-breach rules, which are about a different duty.

,P1 Bi-Encoder,cosine,P2 Cross-Encoder,CE score,P3 BM25,BM25 score
rank,,,,,,
1,Art. 33(1-2),0.396,Recital 85,-6.59,Recital 100,8.0
2,Recital 85,0.396,Art. 33(1-2),-8.44,Recital 32,7.8
3,Art. 70(1),0.393,Art. 16 << GOLD,-10.21,Art. 14(5),7.8
4,Art. 33(3-4),0.386,Art. 34(1-2),-10.77,Recital 35,7.7
5,Recital 85,0.386,Art. 33(3-4),-10.85,Art. 13(2),7.5
6,Art. 34(1-2),0.385,Recital 85,-11.02,Art. 4,7.3
7,Recital 88,0.382,Recital 88,-11.09,Art. 15(1-2),7.1
8,Art. 13(2),0.375,Art. 70(1),-11.11,Art. 14(2),7.1
9,Art. 16 << GOLD,0.367,Art. 13(2),-11.14,Art. 47(2),7.0


**Answer 1** (Bi-Encoder Top-3: Art. 33(1-2), Recital 85, Art. 70(1))

> The context does not directly address how quickly a company must fix wrong information about you. It discusses notifying authorities about personal data breaches but does not specify a timeframe for correcting inaccurate information. [Art. 33(1-2)]

**Answer 2** (Cross-Encoder Top-3: Recital 85, Art. 33(1-2), Art. 16)

> According to GDPR Article 16(1), a company must fix wrong information about you without undue delay. However, there is no specific timeframe mentioned for how quickly this must be done. [Art. 16(1)]

**Answer 3** (BM25 Top-3: Recital 100, Recital 32, Art. 14(5))

> The provided context does not answer this question.

### Answer 1 vs Answer 2 vs Answer 3

| | Gold rank (P1 / P2 / P3) | Answer 1 (Bi-Encoder) | Answer 2 (Bi-Encoder + Cross-Encoder) | Answer 3 (BM25) |
|---|---|---|---|---|
| **Q1** keyword trap | #4 / #1 / not in Top-10 | **Misstated the rule:** "2% of turnover" (the lower tier, Art. 83(4)). | **Correct:** 4% of turnover or EUR 20,000,000, whichever is higher [Art. 83(5)]. | **No answer:** BM25 found unrelated chunks (the law says "fines", not "penalty"). |
| **Q2** exception | #4 / #1 / #3 | **Omitted the exception.** | **Correct:** telling users is not required if the data were encrypted [Art. 34(3)(a)]. | **Wrong, contradicts itself:** "Yes, you have to notify", then explains why encryption means it is not required. The right chunk was only 3rd in its context. |
| **Q3** negation / role | #5 / #1 / #5 | **Made-up reason** (secrecy, Art. 38(5)). | **Correct:** cannot be dismissed or penalised for performing their tasks [Art. 38(3)]. | **Made-up reason** (conflict of interest, Art. 38(6)). |
| **Q4** buried at rank 9 | #9 / #1 / not in Top-10 | **Omission + wrong article** (Art. 6(1)(e)). | **Correct:** must stop unless there are compelling legitimate grounds [Art. 21(1)]. | **Partly correct:** right idea ("no overriding legitimate grounds"), but taken from the right-to-erasure article, Art. 17(1), not Art. 21(1). |
| **Q5** synonym trap | #9 / #3 / not in Top-10 | **No answer.** | **Correct:** corrected "without undue delay" [Art. 16]. | **No answer:** none of the question's words ("quickly", "fix", "wrong") are in Art. 16. |

Only Pipeline 2 (Cross-Encoder re-ranking) gave the right answer in all five cases. BM25 put the gold chunk in its Top-3 only once (Q2), and for questions written in everyday words (Q1, Q4, Q5) it did not find it in the Top-10 at all, because BM25 can only match the exact words used in the law.

## 9. Rank-shift summary

In [18]:
shift = pd.DataFrame([{"id": r["id"], "failure type": r["type"], "gold chunk": r["gold_ref"],
                       "P1 Bi-Encoder rank": r["base_rank"], "P2 Cross-Encoder rank": r["rr_rank"],
                       "P3 BM25 rank": r["bm_rank"] if r["bm_rank"] else "not in Top-10",
                       "gold reaches LLM (P1 / P2 / P3)": " / ".join("yes" if x and x <= TOP_N else "no"
                                                                     for x in (r["base_rank"], r["rr_rank"], r["bm_rank"]))}
                      for r in results]).set_index("id")
shift

,failure type,gold chunk,P1 Bi-Encoder rank,P2 Cross-Encoder rank,P3 BM25 rank,gold reaches LLM (P1 / P2 / P3)
id,,,,,,
Q1,Keyword trap / distractor,Art. 83(5),4,1,not in Top-10,no / yes / no
Q2,Exception / corner case,Art. 34(3),4,1,3,no / yes / yes
Q3,Negation / role constraint,Art. 38(1-3),5,1,5,no / yes / no
Q4,Other - buried at rank 4-10,Art. 21(1-2),9,1,not in Top-10,no / yes / no
Q5,Keyword trap / synonym,Art. 16,9,3,not in Top-10,no / yes / no


## 10. Latency profiling

We time each step with `time.perf_counter()`. Retrieval is repeated 10 times per query (after one warm-up run) to get a stable average; LLM time comes from the 15 answers generated above.

In [19]:
for r in results:                       # warm-up
    baseline_retrieve(r["query"])
times = []
for _ in range(10):
    for r in results:
        t0 = time.perf_counter()
        ids, _ = baseline_retrieve(r["query"])
        t1 = time.perf_counter()
        rerank(r["query"], ids)
        t2 = time.perf_counter()
        bm25_retrieve(r["query"])
        t3 = time.perf_counter()
        times.append({"bi": (t1 - t0) * 1e3, "ce": (t2 - t1) * 1e3, "bm": (t3 - t2) * 1e3})
times = pd.DataFrame(times)
gen1 = np.mean([r["p1"]["gen_s"] for r in results]) * 1e3
gen2 = np.mean([r["p2"]["gen_s"] for r in results]) * 1e3
gen3 = np.mean([r["p3"]["gen_s"] for r in results]) * 1e3
bi, ce, bm = times.bi.mean(), times.ce.mean(), times.bm.mean()

latency = pd.DataFrame({
    "Pipeline 1 (Bi-Encoder)": [bi, 0.0, 0.0, bi, gen1, bi + gen1],
    "Pipeline 2 (Bi-Encoder + Cross-Encoder)": [bi, 0.0, ce, bi + ce, gen2, bi + ce + gen2],
    "Pipeline 3 (BM25)": [0.0, bm, 0.0, bm, gen3, bm + gen3],
}, index=["Bi-Encoder retrieval, Top-10 (ms)", "BM25 retrieval, Top-10 (ms)", "Cross-Encoder re-rank, 10 pairs (ms)",
          "Total retrieval (ms)", "LLM answer (ms)", "End-to-end (ms)"]).round(1)

per_pair = times.ce.mean() / TOP_K
print(f"Cross-Encoder cost: ~{per_pair:.0f} ms per (query, chunk) pair -> scoring all {len(chunks)} chunks "
      f"would take ~{per_pair * len(chunks) / 1e3:.0f} s per query.")
latency

Cross-Encoder cost: ~29 ms per (query, chunk) pair -> scoring all 559 chunks would take ~16 s per query.


,Pipeline 1 (Bi-Encoder),Pipeline 2 (Bi-Encoder + Cross-Encoder),Pipeline 3 (BM25)
"Bi-Encoder retrieval, Top-10 (ms)",12.3,12.3,0.0
"BM25 retrieval, Top-10 (ms)",0.0,0.0,2.5
"Cross-Encoder re-rank, 10 pairs (ms)",0.0,294.0,0.0
Total retrieval (ms),12.3,306.3,2.5
LLM answer (ms),58538.5,55421.0,55297.5
End-to-end (ms),58550.7,55727.2,55299.9


In [20]:
# save the results used in the report
shift.to_csv(RESULTS_DIR / "rank_shifts.csv")
latency.to_csv(RESULTS_DIR / "latency.csv")
(RESULTS_DIR / "answers.json").write_text(json.dumps([{
    "id": r["id"], "type": r["type"], "query": r["query"], "gold": r["gold_ref"], "correct": r["correct"],
    "normal_rank": r["base_rank"], "cross_encoder_rank": r["rr_rank"], "bm25_rank": r["bm_rank"],
    "normal_top10": [chunk_df.ref[i] for i in r["base_ids"]],
    "cross_encoder_top10": [chunk_df.ref[i] for i in r["rr_ids"]],
    "bm25_top10": [chunk_df.ref[i] for i in r["bm_ids"]],
    "answer_1": r["p1"]["answer"], "answer_2": r["p2"]["answer"], "answer_3": r["p3"]["answer"]}
    for r in results], indent=2, ensure_ascii=False))
print(sorted(p.name for p in RESULTS_DIR.iterdir()))

['answers.json', 'latency.csv', 'rank_shifts.csv']


## Summary

* **Rank shifts:** the Cross-Encoder moved the gold chunk into the **Top-3** in all five failure queries (rank 1 in four of them). The Bi-Encoder never had it in the Top-3 (rank 4-9). BM25 had it in the Top-3 only for Q2 and missed it completely (not in Top-10) for Q1, Q4 and Q5.
* **Answers:** Answer 2 was correct in all five. Answers 1 and 3 misstated a rule, missed an exception, made up a reason, or could not answer (Answer 3 was partly correct for Q4).
* **Easy question** (same words as the law, section 11): all three pipelines found Art. 33(1) first and all three answers were correct.
* **Latency:** BM25 is the fastest retriever (~2.5 ms), the Bi-Encoder takes ~12 ms, and Cross-Encoder re-ranking adds ~0.3 s (about 25x slower retrieval), which is still under 1% of the time the LLM takes to write an answer.

## 11. Try more questions

`try_question()` runs all three pipelines for any question and shows the Bi-Encoder, Cross-Encoder and BM25 rankings side by side, then Answers 1, 2 and 3.
To try your own, write `try_question("your question")` in a new cell (each question takes about 2-3 minutes on CPU, because the LLM writes three answers).

In [21]:
def try_question(query):
    res = run_pipelines(query)
    display(Markdown(f"### {query}"))
    display(pd.DataFrame({"P1 Bi-Encoder": [chunk_df.ref[i] for i in res["base_ids"]],
                          "P2 Cross-Encoder": [chunk_df.ref[i] for i in res["rr_ids"]],
                          "P3 BM25": [chunk_df.ref[i] for i in res["bm_ids"]]},
                         index=pd.Index(range(1, TOP_K + 1), name="rank")))
    display(Markdown(f"**Answer 1** (Bi-Encoder Top-3):\n\n> {res['p1']['answer']}\n\n"
                     f"**Answer 2** (Cross-Encoder Top-3):\n\n> {res['p2']['answer']}\n\n"
                     f"**Answer 3** (BM25 Top-3):\n\n> {res['p3']['answer']}"))

In [22]:
try_question("Within how many hours must a personal data breach be reported to the supervisory authority?")

### Within how many hours must a personal data breach be reported to the supervisory authority?

,P1 Bi-Encoder,P2 Cross-Encoder,P3 BM25
rank,,,
1,Art. 33(1-2),Art. 33(1-2),Art. 33(1-2)
2,Recital 85,Recital 85,Recital 85
3,Recital 87,Art. 33(5),Art. 33(3-4)
4,Art. 33(3-4),Art. 33(3-4),Art. 34(4)
5,Art. 33(5),Recital 87,Art. 33(5)
6,Art. 34(4),Art. 34(4),Recital 86
7,Art. 70(1),Recital 86,Art. 54(1)
8,Art. 40(2),Art. 40(2),Recital 87
9,Recital 86,Recital 85,Art. 34(1-2)


**Answer 1** (Bi-Encoder Top-3):

> A personal data breach must be reported to the supervisory authority within 72 hours after becoming aware of it [Art. 33(1)].

**Answer 2** (Cross-Encoder Top-3):

> A personal data breach must be reported to the supervisory authority within 72 hours after becoming aware of it [Art. 33(1-2)].

**Answer 3** (BM25 Top-3):

> A personal data breach must be reported to the supervisory authority within 72 hours after becoming aware of it [Art. 33(1)].

In [23]:
try_question("Can data subjects object to processing for scientific research?")

### Can data subjects object to processing for scientific research?

,P1 Bi-Encoder,P2 Cross-Encoder,P3 BM25
rank,,,
1,Recital 33,Art. 21(6),Recital 156
2,Recital 157,Recital 33,Recital 157
3,Recital 46,Recital 157,Recital 33
4,Recital 156,Recital 156,Art. 89(1)
5,Recital 50,Recital 159,Art. 21(6)
6,Recital 159,Recital 62,Recital 159
7,Art. 21(6),Recital 69,Recital 156
8,Recital 63,Recital 63,Art. 9(2)
9,Recital 62,Recital 50,Recital 162


**Answer 1** (Bi-Encoder Top-3):

> The provided context does not explicitly state whether data subjects can object to processing for scientific research. [Recital 33] discusses giving consent for certain areas of scientific research but does not mention objections.

**Answer 2** (Cross-Encoder Top-3):

> Yes, data subjects can object to processing for scientific research [Art. 21(6)]. However, this applies only where personal data are processed for scientific or historical research purposes or statistical purposes pursuant to Article 89(1). Additionally, data subjects should have the opportunity to give their consent only to certain areas of research or parts of research projects to the extent allowed by the intended purpose [Recital 33].

**Answer 3** (BM25 Top-3):

> Yes, data subjects can object to processing for scientific research [Recital 157]. However, GDPR allows for specific conditions and safeguards, including allowing data subjects to give consent to certain areas of scientific research [Recital 33].

In [24]:
try_question("When is a data protection impact assessment required?")

### When is a data protection impact assessment required?

,P1 Bi-Encoder,P2 Cross-Encoder,P3 BM25
rank,,,
1,Recital 91,Recital 91,Art. 35(4-5)
2,Art. 35(10-11),Art. 35(3),Recital 91
3,Art. 35(1-2),Recital 90,Art. 35(1-2)
4,Art. 35(8-9),Recital 91,Art. 35(10-11)
5,Recital 90,Art. 35(4-5),Art. 35(3)
6,Recital 92,Art. 35(1-2),Recital 84
7,Art. 35(4-5),Art. 35(8-9),Art. 35(8-9)
8,Art. 35(3),Recital 92,Recital 90
9,Art. 36(1),Art. 36(1),Art. 35(7)


**Answer 1** (Bi-Encoder Top-3):

> A data protection impact assessment is required when processing is likely to result in a high risk to the rights and freedoms of data subjects, especially when using optic-electronic devices or for large-scale monitoring operations [Recital 91]. It is also required when the processing involves personal data from patients or clients by an individual physician, other health care professional or lawyer, unless the processing is not on a large scale [Recital 91]. Additionally, an assessment is needed before processing activities if the processing is based on Union law or national law and a previous impact assessment was done for the legal basis [Art. 35(10-11)].

**Answer 2** (Cross-Encoder Top-3):

> A data protection impact assessment is required when processing involves monitoring publicly accessible areas on a large scale, especially with the use of optic-electronic devices, or when decisions based on automated processing significantly affect individuals [Recital 91]. It's also required for processing on a large scale of special categories of data or criminal convictions, or systematic monitoring of a publicly accessible area on a large scale [Art. 35(3)].

**Answer 3** (BM25 Top-3):

> A data protection impact assessment is required when the processing is likely to result in a high risk to the rights and freedoms of data subjects [Art. 35(1)]. This risk consideration includes monitoring publicly accessible areas on a large scale, especially with the use of optic-electronic devices, or processing that prevents data subjects from exercising rights or using services [Recital 91]. Notably, it is not mandatory for processing personal data from patients or clients by individual healthcare professionals [Recital 91].